## Phase 1: Library Imports and Data Cleaning
Load the dataset and transform the list of strings into a One-Hot matrix for the Apriori algorithm.

In [16]:
import sys
import subprocess

# Install missing packages if needed
for pkg in ['mlxtend', 'pyECLAT']:
    try:
        __import__(pkg)
    except ImportError:
        print(f'Installing missing package: {pkg}')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

import pandas as pd
import numpy as np
import ast
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
import matplotlib.pyplot as plt
import seaborn as sns

ImportError: cannot import name '_slice' from 'numpy._core.umath' (/Users/joanaanmartins/opt/anaconda3/envs/PDS/lib/python3.11/site-packages/numpy/_core/umath.py)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math
from matplotlib.patches import Patch
import csv

# Mlxtend library
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules
from mlxtend.preprocessing import TransactionEncoder

# pyECLAT library
from pyECLAT import ECLAT

# sklearn libraries
from sklearn.model_selection import train_test_split

# Utils package
from utils import (
    count_items,
    generate_string_pairs,
    count_tuple_occurrences,
    generate_string_triplets
)

# Matplotlib Options
plt.rcParams['figure.figsize'] = [16, 8]
font = {'weight' : 'bold',
        'size'   : 14}
plt.rc('font', **font)

# Pandas options
pd.set_option("display.max_columns", 100)
pd.set_option('display.max_colwidth', None)

ModuleNotFoundError: No module named 'utils'

In [3]:
df_basket = pd.read_csv('customer_basket (1).csv')


In [ ]:
df_basket.head()

,invoice_id,list_of_goods,customer_id
0,3700630,"['chicken', 'rice', 'pepper', 'whole wheat ric...",12912
1,10242376,"['low fat yogurt', 'tomatoes', 'pepper', 'aspa...",22853
2,91550,"['cake', 'tomatoes', 'pancakes', 'iPad', 'fina...",19
3,3137503,"['cereals', 'megaman zero', 'final fantasy XIX...",10995
4,7165061,"['rice', 'frozen smoothie', 'black tea', 'tea'...",27807


In [20]:


print("Processing lists of products...")
df_basket['list_of_goods'] = df_basket['list_of_goods'].apply(ast.literal_eval)

print("Creating the transaction matrix (One-Hot Encoding)...")
te = TransactionEncoder()
te_ary = te.fit(df_basket['list_of_goods']).transform(df_basket['list_of_goods'])

df_transactions = pd.DataFrame(te_ary, columns=te.columns_)

print(f"Data ready! {df_transactions.shape[0]} transactions and {df_transactions.shape[1]} unique products.")
display(df_transactions.head(3))

Processing lists of products...
Creating the transaction matrix (One-Hot Encoding)...


NameError: name 'TransactionEncoder' is not defined

## Phase 2: Frequent Pattern Extraction (Apriori)
Extract combinations that appear in at least 1% of the transactions. We use low_memory=True to ensure stability with 100,000 rows.

In [ ]:
print("Running the Apriori algorithm...")
frequent_itemsets = apriori(df_transactions, min_support=0.01, use_colnames=True, low_memory=True)

print(f"Found {len(frequent_itemsets)} frequent itemsets.")
display(frequent_itemsets.sort_values('support', ascending=False).head(10))

## Phase 3: Generation of Association Rules
Discover rules with a Lift greater than 1.0 (which indicates a true positive association between the products).

In [ ]:
print("Generating association rules...")
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)

print(f"Generated {len(rules)} rules with Lift >= 1.0")
display(rules.sort_values('lift', ascending=False).head(10))

## Phase 4: Visualization and Insights
Scatter plot of Support vs Confidence to visualize the impact of Lift.

In [ ]:
plt.figure(figsize=(10, 6))
scatter = plt.scatter(rules['support'], rules['confidence'], c=rules['lift'], cmap='viridis', alpha=0.8)
plt.colorbar(scatter, label='Lift')
plt.xlabel('Support')
plt.ylabel('Confidence')
plt.title('Association Rules: Support vs Confidence')
plt.tight_layout()
plt.show()

## Phase 5: Cross-cluster Basket Analysis
Integrate cluster labels from `dataset_clusters.csv` with the `customer_basket` transactions using `customer_info.csv` as a bridge. Then analyze product frequencies and association rules per cluster.

In [ ]:
# Load cluster labels and customer mapping
try:
    from mlxtend.preprocessing import TransactionEncoder
except ImportError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "mlxtend"])
    from mlxtend.preprocessing import TransactionEncoder

print('Loading cluster labels and customer mapping...')
df_clusters = pd.read_csv('dataset_clusters.csv')
df_customers = pd.read_csv('customer_info.csv')

cluster_cols = ['customer_name', 'Cluster', 'cluster']
if 'customer_name' not in df_clusters.columns:
    raise ValueError('dataset_clusters.csv must contain customer_name')
df_clusters = df_clusters[cluster_cols]

df_customers = df_customers[['customer_id', 'customer_name']]

df_basket_clusters = df_basket.merge(df_customers, on='customer_id', how='left')
df_basket_clusters = df_basket_clusters.merge(df_clusters, on='customer_name', how='left')

print('Merged baskets with customer info and cluster labels:')
print('  transactions:', len(df_basket_clusters))
print('  missing customer_name:', df_basket_clusters['customer_name'].isna().sum())
print('  missing cluster labels:', df_basket_clusters['Cluster'].isna().sum())
display(df_basket_clusters.head())

# Explore basket coverage by cluster
cluster_coverage = (
    df_basket_clusters.groupby('Cluster')
    .agg(transactions=('invoice_id', 'nunique'),
         customers=('customer_id', 'nunique'))
    .reset_index()
    .sort_values('transactions', ascending=False)
)
print('Customer basket coverage by cluster:')
display(cluster_coverage)

# Explode products for cluster frequency analysis
print('Computing top products per cluster...')
records = []
for _, row in df_basket_clusters[df_basket_clusters['Cluster'].notna()].iterrows():
    if isinstance(row['list_of_goods'], list):
        for prod in row['list_of_goods']:
            records.append({'Cluster': row['Cluster'], 'product': prod})

if records:
    df_products = pd.DataFrame(records)
    top_products_by_cluster = (
        df_products.groupby(['Cluster', 'product'])
        .size()
        .reset_index(name='count')
        .sort_values(['Cluster', 'count'], ascending=[True, False])
    )

    for cluster_label, group in top_products_by_cluster.groupby('Cluster'):
        print(f'Cluster {cluster_label} top products:')
        display(group.head(10))
else:
    print('No cluster-labeled basket records found to analyze.')

# Generate cluster-specific association rules
print('Generating cluster-specific association rules...')
for cluster_label in sorted(df_basket_clusters['Cluster'].dropna().unique()):
    df_cluster = df_basket_clusters[df_basket_clusters['Cluster'] == cluster_label]
    if len(df_cluster) < 20:
        print(f'  Skipping cluster {cluster_label} because it has only {len(df_cluster)} baskets.')
        continue

    te_cluster = TransactionEncoder()
    te_ary_cluster = te_cluster.fit(df_cluster['list_of_goods']).transform(df_cluster['list_of_goods'])
    cluster_transactions = pd.DataFrame(te_ary_cluster, columns=te_cluster.columns_)

    frequent_cluster = apriori(cluster_transactions, min_support=0.02, use_colnames=True, low_memory=True)
    if frequent_cluster.empty:
        print(f'  No frequent itemsets found for cluster {cluster_label}.')
        continue

    rules_cluster = association_rules(frequent_cluster, metric='lift', min_threshold=1.0)
    if rules_cluster.empty:
        print(f'  No association rules found for cluster {cluster_label}.')
        continue

    print(f'Cluster {cluster_label} top rules:')
    display(rules_cluster.sort_values('lift', ascending=False).head(10))

Loading cluster labels and customer mapping...
Merged baskets with customer info and cluster labels:
  transactions: 103349
  missing customer_name: 0
  missing cluster labels: 0


,invoice_id,list_of_goods,customer_id,customer_name,Cluster,cluster
0,3700630,"['chicken', 'rice', 'pepper', 'whole wheat ric...",12912,Msc. George Davis,2,2
1,10242376,"['low fat yogurt', 'tomatoes', 'pepper', 'aspa...",22853,Patricia Silva,0,0
2,91550,"['cake', 'tomatoes', 'pancakes', 'iPad', 'fina...",19,Phd. Evelyn Quinn,2,2
3,3137503,"['cereals', 'megaman zero', 'final fantasy XIX...",10995,Msc. David Stupar,0,0
4,7165061,"['rice', 'frozen smoothie', 'black tea', 'tea'...",27807,Noel Froelich,0,0


Customer basket coverage by cluster:


,Cluster,transactions,customers
0,0,48757,15078
4,4,18797,3580
2,2,13669,4272
3,3,12189,2997
1,1,8729,2805


Computing top products per cluster...
No cluster-labeled basket records found to analyze.
Generating cluster-specific association rules...
